In [ ]:
import pandas as pd
import psycopg2
import os
import cenpy
from dotenv import load_dotenv



C:\Users\Syed Haque\AppData\Roaming\Python\Python314\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [ ]:
load_dotenv("../.env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

acs = cenpy.products.ACS()

In [2]:
conn = psycopg2.connect(
    host="awesome-hw.sdsc.edu",
    port=5432,
    dbname="nourish",
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

def query_db(query):
    """Query the nourish database and return data as a `pd.Dataframe`"""
    try:
        with conn.cursor() as cursor:
            cursor.execute(query)
            columns = [desc[0] for desc in cursor.description]
            rows = cursor.fetchall()

        return pd.DataFrame(rows, columns=columns)
    except Exception as e:
        conn.rollback()
        raise e

query_db("SELECT 1")
print(f"Connection successful")

Connection successful


In [3]:
def save_df_to_json(df: pd.DataFrame, filename: str):
    """Save a pandas DataFrame to a JSON file."""
    with open(f"../data/nodes/{filename}", "w") as f:
        df.to_json(f, index=False, orient="records", indent=2)

    print(f"Data saved to data/nodes/{filename}")

##### Entity 1: `State`

In [ ]:
query = """
    SELECT
    1 as id,
    'CA' as code,
    'California' as name
"""

state_df = query_db(query)
save_df_to_json(state_df, "state.json")
print(f"Rows: {state_df.shape[0]}, Columns: {state_df.shape[1]}")
state_df.head()

##### Entity 2: `County`

In [ ]:
query= """
SELECT id, county as name
FROM county_neighborhoods
"""

county_df = query_db(query)
save_df_to_json(county_df, "county.json")
print(f"Rows: {county_df.shape[0]}, Columns: {county_df.shape[1]}")
county_df.head()

##### Entity 3: `City`

In [ ]:
query = """
WITH cte AS (
    SELECT
        c.id,
        c.city,
        ST_Union(op.way) AS geom,  -- merge all polygons for that city
        MIN(ons.osm_id) AS osm_id  -- arbitrary representative ID
    FROM
        city_neighborhoods c
    LEFT JOIN osm_planet_socal_2025.osn_names ons
        ON c.city = ons.name
        AND ons.geom_type = 'polygon'
    LEFT JOIN osm_planet_socal_2025.planet_osm_polygon op
        ON ons.osm_id = op.osm_id
        AND ons.geom_type = 'polygon'
    WHERE
        op.osm_id < 0
        AND county = 'San Diego'
    GROUP BY
        c.city,
        c.id
)
SELECT
    id,
    city as name,
    ST_Transform(geom, 4326) AS geom,
    ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt,
    ST_AsText(ST_Centroid(ST_Transform(geom, 4326))) AS centroid_wkt
FROM cte; 
"""


city_df = query_db(query)



# save_df_to_json(city_df, "city.json")
# print(f"Rows: {city_df.shape[0]}, Columns: {city_df.shape[1]}")


InterfaceError: connection already closed

In [ ]:
# #Add city attributes
cities_names_list = city_df['name'].unique()

attribute_variables = ['B01003_001E','B11001_001E']
cities_demographic_data = [['name', "total_population",'total_households']]

for city in cities_names_list:
  city_data =[]
  place_name = city + ', CA'
  # city_gdf = ox.geocode_to_gdf(place_name)
  # city_geometry = city_gdf.geometry.iloc[0]

  city_pop_gdf = acs.from_place(place_name, variables=attribute_variables)
  city_pop_tot = city_pop_gdf['B01003_001E'].sum()
  city_tot_household = city_pop_gdf['B11001_001E'].sum()
  city_data = [city, city_pop_tot, city_tot_household]
  cities_demographic_data.append(city_data)

cities_dem_df = pd.DataFrame(cities_demographic_data)
cities_dem_df.columns = cities_dem_df.iloc[0]
cities_dem_df = cities_dem_df[1:]
merged_city_df = pd.merge(city_df, cities_dem_df, on='name', how='inner')

print(f"Rows: {merged_city_df.shape[0]}, Columns: {merged_city_df.shape[1]}")

merged_city_df.head()

save_df_to_json(merged_city_df, "city.json")
print(f"Rows: {merged_city_df.shape[0]}, Columns: {merged_city_df.shape[1]}")



Matched: Carlsbad, CA to Carlsbad city within layer Incorporated Places
Matched: Chula Vista, CA to Chula Vista city within layer Incorporated Places
Matched: Coronado, CA to Coronado city within layer Incorporated Places
Matched: Del Mar, CA to Del Mar city within layer Incorporated Places
Matched: El Cajon, CA to El Cajon city within layer Incorporated Places
Matched: Encinitas, CA to Encinitas city within layer Incorporated Places
Matched: Escondido, CA to Escondido city within layer Incorporated Places
Matched: Imperial Beach, CA to Imperial Beach city within layer Incorporated Places
Matched: La Mesa, CA to La Mesa city within layer Incorporated Places
Matched: Lemon Grove, CA to Lemon Grove city within layer Incorporated Places
Matched: National City, CA to National City city within layer Incorporated Places
Matched: Oceanside, CA to Oceanside city within layer Incorporated Places
Matched: Poway, CA to Poway city within layer Incorporated Places
Matched: San Diego, CA to San Dieg

C:\Users\Syed Haque\AppData\Roaming\Python\Python314\site-packages\cenpy\products.py:993: UserWarning: Cannot disambiguate placename Spring Valley. Picking the shortest, best matched placename, Spring Valley CDP, from Spring Valley CDP, Spring Valley CDP
  warn(


Matched: Spring Valley, CA to Spring Valley CDP within layer Census Designated Places
Matched: Valley Center, CA to Valley Center CDP within layer Census Designated Places
Matched: Winter Gardens, CA to Winter Gardens CDP within layer Census Designated Places
Rows: 52, Columns: 7


,id,name,geom,geom_wkt,centroid,total_population,total_households
0,8,Carlsbad,0103000020E6100000010000001D0300000EC40D53365A...,"POLYGON((-117.4095657 33.1325654995874,-117.40...",0101000020E610000049A785B9EB535DC041F8FAFE4F8F...,34356.0,13245.0
1,9,Chula Vista,0103000020E610000002000000170600009710BDD6EF47...,"POLYGON((-117.1240136 32.6463658995912,-117.12...",0101000020E61000003AB1DC6FEC405DC0D69967816650...,150885.0,44892.0
2,10,Coronado,0103000020E6100000020000007A0100000DF3D4D97F4E...,"POLYGON((-117.2265534 32.6903901995905,-117.22...",0101000020E61000000A7445D09F4A5DC054781F833252...,7363.0,2991.0
3,11,Del Mar,0103000020E610000001000000CD000000C7EDE1DC7051...,"POLYGON((-117.2725136 32.9801386995877,-117.27...",0101000020E610000023007032CF505DC09BE5A83B4C7B...,0.0,0.0
4,12,El Cajon,0103000020E61000000100000013090000DE2230D6B740...,"POLYGON((-117.0112205 32.8202865995889,-117.01...",0101000020E61000002CD85DA9783D5DC01DEE5A859D66...,49447.0,15381.0


##### Entity 4: `Community`

In [ ]:
query = """
SELECT id, community as name
FROM community_neighborhoods
WHERE county = 'San Diego';
"""

community_df = query_db(query)
save_df_to_json(community_df, "community.json")
print(f"Rows: {community_df.shape[0]}, Columns: {community_df.shape[1]}")
community_df.head()

##### Entity 5: `Zipcode`

In [ ]:
query = """
WITH san_diego_zipcodes AS (

SELECT DISTINCT CAST(unnest(zipcodes) AS TEXT) as zipcode
FROM city_neighborhoods
WHERE county = 'San Diego'

UNION

SELECT DISTINCT CAST(unnest(zipcodes) AS TEXT) as zipcode
FROM community_neighborhoods
WHERE county = 'San Diego'
)

SELECT

t.zipcode,
ST_Transform(s.geom, 4326) AS geom,
ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt,
ST_AsText(ST_Centroid(ST_Transform(geom, 4326))) AS centroid_wkt

FROM
san_diego_zipcodes t
JOIN
test_zipcodes s ON t.zipcode = s.zip::text;
"""


zipcode_df = query_db(query)
zipcode_df["id"] = zipcode_df.index + 1
zipcode_df = zipcode_df[["id"]+[col for col in zipcode_df.columns if col != "id"]]
save_df_to_json(zipcode_df, "zipcode.json")
print(f"Rows: {zipcode_df.shape[0]}, Columns: {zipcode_df.shape[1]}")
zipcode_df.head()

##### Entity 6: `BusinessLocation`

In [ ]:
query = """
SELECT    
    id,
    name,
    url,
    address,
    city,
    zip,
    latitude,
    longitude,
    blockgroup,
    categories,
    avg_rating,
    franchise,
    confidence,
    reasoning,
    ST_Transform(geom, 4326) AS geom,
    ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt
FROM ca_businesses_with_ai_franchise
"""

business_location_df = query_db(query)
business_location_df[business_location_df["zip"].isin(zipcode_df["zipcode"].values)]
avg_rating_fill = business_location_df["avg_rating"].mean()
business_location_df["avg_rating"] = business_location_df["avg_rating"].fillna(avg_rating_fill)
save_df_to_json(business_location_df, "business_location.json")
print(f"Rows: {business_location_df.shape[0]}, Columns: {business_location_df.shape[1]}")
business_location_df.head()

##### Entity 7: `Business`

In [ ]:
business_df = business_location_df[["name"]].drop_duplicates().reset_index(drop=True)
business_df["id"] = business_df.index + 1
business_df = business_df[["id"]+[col for col in business_df.columns if col != "id"]]
save_df_to_json(business_df, "business.json")
print(f"Rows: {business_df.shape[0]}, Columns: {business_df.shape[1]}")
business_df.head()

##### Entity 8: `BlockGroup`

In [ ]:
query = """
SELECT
    sbg.ctblockgroup as id,
    bd.std_geography_id AS geo_id,
    sbg.ctblockgroup,

    -- INCOME
    imp.AVGDI_CY as average_income,
    imp.MEDDI_CY as median_income,
    imp.di0_cy    AS income_under_15000,
    imp.di15_cy   AS income_15000_24999,
    imp.di25_cy   AS income_25000_34999,
    imp.di35_cy   AS income_35000_49999,
    imp.di50_cy   AS income_50000_74999,
    imp.di75_cy   AS income_75000_99999,
    imp.di100_cy  AS income_100000_149999,
    imp.di150_cy  AS income_150000_199999,
    imp.di200_cy  AS income_200000_plus,

    -- POPULATION
    imp.TOTPOP_CY as total_population,

    -- AGE
    (imp.male0 + imp.male5 + imp.fem0 + imp.fem5) AS population_0_9,
    (imp.male10 + imp.male15 + imp.fem10 + imp.fem15) AS population_10_19,
    (imp.male20 + imp.male25 + imp.fem20 + imp.fem25) AS population_20_29,
    (imp.male30 + imp.male35 + imp.fem30 + imp.fem35) AS population_30_39,
    (imp.male40 + imp.male45 + imp.fem40 + imp.fem45) AS population_40_49,
    (imp.male50 + imp.male55 + imp.fem50 + imp.fem55) AS population_50_59,
    (imp.male60 + imp.male65 + imp.fem60 + imp.fem65) AS population_60_69,
    (imp.male70 + imp.male75 + imp.fem70 + imp.fem75) AS population_70_79,
    (imp.male80 + imp.male85 + imp.fem80 + imp.fem85) AS population_80_plus,

    -- MALE
    (imp.male0  + imp.male5)  AS male_population_0_9,
    (imp.male10 + imp.male15) AS male_population_10_19,
    (imp.male20 + imp.male25) AS male_population_20_29,
    (imp.male30 + imp.male35) AS male_population_30_39,
    (imp.male40 + imp.male45) AS male_population_40_49,
    (imp.male50 + imp.male55) AS male_population_50_59,
    (imp.male60 + imp.male65) AS male_population_60_69,
    (imp.male70 + imp.male75) AS male_population_70_79,
    (imp.male80 + imp.male85) AS male_population_80_plus,
    (imp.male0  + imp.male5  +
     imp.male10 + imp.male15 +
     imp.male20 + imp.male25 +
     imp.male30 + imp.male35 +
     imp.male40 + imp.male45 +
     imp.male50 + imp.male55 +
     imp.male60 + imp.male65 +
     imp.male70 + imp.male75 +
     imp.male80 + imp.male85) AS male_population_total,

    -- FEMALE
    (imp.fem0  + imp.fem5)  AS female_population_0_9,
    (imp.fem10 + imp.fem15) AS female_population_10_19,
    (imp.fem20 + imp.fem25) AS female_population_20_29,
    (imp.fem30 + imp.fem35) AS female_population_30_39,
    (imp.fem40 + imp.fem45) AS female_population_40_49,
    (imp.fem50 + imp.fem55) AS female_population_50_59,
    (imp.fem60 + imp.fem65) AS female_population_60_69,
    (imp.fem70 + imp.fem75) AS female_population_70_79,
    (imp.fem80 + imp.fem85) AS female_population_80_plus,
    (imp.fem0  + imp.fem5  +
     imp.fem10 + imp.fem15 +
     imp.fem20 + imp.fem25 +
     imp.fem30 + imp.fem35 +
     imp.fem40 + imp.fem45 +
     imp.fem50 + imp.fem55 +
     imp.fem60 + imp.fem65 +
     imp.fem70 + imp.fem75 +
     imp.fem80 + imp.fem85) AS female_population_total,

    -- BUSINESS COUNTS
    bd.s01_bus  AS businesses_total_sic,
    bd.s02_bus  AS businesses_agriculture_mining_sic,
    bd.s08_bus  AS businesses_wholesale_trade_sic,
    bd.s09_bus  AS businesses_retail_trade_sic,
    bd.s12_bus  AS businesses_food_stores_sic,
    bd.s13_bus  AS businesses_auto_gas_sic,
    bd.s16_bus  AS businesses_eating_drinking_sic,
    bd.s17_bus  AS businesses_misc_retail_sic,
    bd.s18_bus  AS businesses_finance_insurance_realestate_sic,
    bd.s19_bus  AS businesses_banks_sic,
    bd.s20_bus  AS businesses_securities_sic,
    bd.s21_bus  AS businesses_insurance_sic,
    bd.s22_bus  AS businesses_real_estate_sic,
    bd.s23_bus  AS businesses_services_sic,
    bd.s24_bus  AS businesses_hotels_sic,
    bd.s25_bus  AS businesses_auto_services_sic,
    bd.s26_bus  AS businesses_amusements_sic,
    bd.s27_bus  AS businesses_health_services_sic,
    bd.s28_bus  AS businesses_legal_services_sic,
    bd.s29_bus  AS businesses_education_sic,
    bd.s30_bus  AS businesses_other_services_sic,

    bd.n01_bus  AS businesses_total_naics,
    bd.n02_bus  AS businesses_agriculture_naics,
    bd.n07_bus  AS businesses_wholesale_trade_naics,
    bd.n08_bus  AS businesses_retail_trade_naics,
    bd.n12_bus  AS businesses_building_materials_naics,
    bd.n13_bus  AS businesses_food_beverage_stores_naics,
    bd.n14_bus  AS businesses_health_personal_care_naics,
    bd.n15_bus  AS businesses_gas_stations_naics,
    bd.n27_bus  AS businesses_real_estate_leasing_naics,
    bd.n35_bus  AS businesses_accommodation_food_naics,
    bd.n36_bus  AS businesses_accommodation_naics,
    bd.n37_bus  AS businesses_food_services_naics,

    -- BUSINESS SALES
    bd.s01_sales AS sales_total_sic,
    bd.s02_sales AS sales_agriculture_mining_sic,
    bd.s08_sales AS sales_wholesale_trade_sic,
    bd.s09_sales AS sales_retail_trade_sic,
    bd.s12_sales AS sales_food_stores_sic,
    bd.s13_sales AS sales_auto_gas_sic,
    bd.s16_sales AS sales_eating_drinking_sic,
    bd.s17_sales AS sales_misc_retail_sic,
    bd.s18_sales AS sales_finance_insurance_realestate_sic,
    bd.s19_sales AS sales_banks_sic,
    bd.s20_sales AS sales_securities_brokers_sic,
    bd.s21_sales AS sales_insurance_sic,
    bd.s22_sales AS sales_real_estate_sic,
    bd.s23_sales AS sales_services_sic,
    bd.s24_sales AS sales_hotels_sic,
    bd.s25_sales AS sales_auto_services_sic,
    bd.s26_sales AS sales_amusement_sic,
    bd.s27_sales AS sales_health_services_sic,
    bd.s28_sales AS sales_legal_services_sic,
    bd.s29_sales AS sales_education_sic,
    bd.s30_sales AS sales_other_services_sic,

    bd.n01_sales AS sales_total_naics,
    bd.n02_sales AS sales_agriculture_naics,
    bd.n07_sales AS sales_wholesale_trade_naics,
    bd.n08_sales AS sales_retail_trade_naics,
    bd.n12_sales AS sales_building_materials_naics,
    bd.n13_sales AS sales_food_beverage_stores_naics,
    bd.n14_sales AS sales_health_personal_care_naics,
    bd.n15_sales AS sales_gas_stations_naics,
    bd.n27_sales AS sales_real_estate_leasing_naics,
    bd.n35_sales AS sales_accommodation_food_naics,
    bd.n36_sales AS sales_accommodation_naics,
    bd.n37_sales AS sales_food_services_naics,

    ST_Transform(sbg.geom, 4326) AS geom,
    ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt,
    ST_AsText(ST_Centroid(ST_Transform(geom, 4326))) AS centroid_wkt

FROM sandag_layer_census_block_groups sbg
LEFT JOIN bgs_sd_imp imp
    ON CAST(sbg.ctblockgroup AS TEXT) = CAST(CONCAT(LTRIM(imp.tractce, '0'), imp.blkgrpce) AS TEXT)
LEFT JOIN esri_business_data bd
    ON TRIM(LEADING '0' FROM SUBSTR(CAST(bd.std_geography_id AS TEXT), 5)) = CAST(sbg.ctblockgroup AS TEXT)
LEFT JOIN esri_consumer_spending_cols cs
    ON TRIM(LEADING '0' FROM SUBSTR(CAST(cs.std_geography_id AS TEXT), 5)) = CAST(sbg.ctblockgroup AS TEXT)
ORDER BY sbg.ctblockgroup ASC;
 
"""

block_group_df = query_db(query)
save_df_to_json(block_group_df, "block_group.json")
print(f"Rows: {block_group_df.shape[0]}, Columns: {block_group_df.shape[1]}")
block_group_df.head() 

In [ ]:
# show rows with duplicated ctblockgrou
block_group_df[block_group_df.duplicated(subset=['ctblockgroup'], keep=False)]

In [ ]:
for c in block_group_df.columns:
    print(c)

##### Entity 9: `Zone Location`

In [ ]:
query = """
SELECT
id,
zone_name,
imp_date,
ordnum,
shape_length,
shape_area,
legend,
ST_Transform(geom, 4326) AS geom,
ST_AsText(ST_Transform(geom, 4326)) AS geom_ewkt,
ST_AsText(ST_Centroid(ST_Transform(geom, 4326))) AS centroid_wkt

FROM sandag_layer_zoning_base_sd_new
"""

zone_location_df = query_db(query)
save_df_to_json(zone_location_df, "zone_location.json")
print(f"Rows: {zone_location_df.shape[0]}, Columns: {zone_location_df.shape[1]}")
zone_location_df.head()

##### Entity 10: `Zone Type`

In [ ]:
query = """
SELECT
DISTINCT(zone_name) as name,
legend
FROM sandag_layer_zoning_base_sd_new
"""

zone_type_df = query_db(query)
zone_type_df["id"] = zone_type_df.index + 1
zone_type_df = zone_type_df[["id"]+[col for col in zone_type_df.columns if col != "id"]]
save_df_to_json(zone_type_df, "zone_type.json")
print(f"Rows: {zone_type_df.shape[0]}, Columns: {zone_type_df.shape[1]}")
zone_type_df.head()